In [1]:
# import libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE
from sklearn.cluster import DBSCAN
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, auc, accuracy_score, precision_score, recall_score, f1_score
import xgboost as xgb
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
import numpy as np

# Load the dataset
df = pd.read_csv('../../data/raw/FraudTest.csv')

In [2]:
# Feature Engineering: Extracting features from the transaction date and time
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['hour'] = df['trans_date_trans_time'].dt.hour
df['day'] = df['trans_date_trans_time'].dt.day
df['day_of_week'] = df['trans_date_trans_time'].dt.dayofweek
df['month'] = df['trans_date_trans_time'].dt.month

In [3]:
# Feature Engineering: Calculating distance between cardholder and merchant locations
def calculate_distance(row):
    lat1, lon1, lat2, lon2 = map(np.radians, [row['lat'], row['long'], row['merch_lat'], row['merch_long']])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    distance = 6371 * c  # Radius of Earth in kilometers
    return distance

df['distance'] = df.apply(calculate_distance, axis=1)

In [4]:
# Selecting relevant features for the model
features = ['amt', 'city_pop', 'hour', 'day', 'day_of_week', 'month', 'distance',
            'merchant', 'category', 'state', 'job', 'gender', 'city']
X = df[features]
y = df['is_fraud']

In [5]:
# Splitting the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Preprocessing pipeline
numerical_features = ['amt', 'city_pop', 'hour', 'day', 'day_of_week', 'month', 'distance']
categorical_features = ['merchant', 'category', 'state', 'job', 'gender', 'city']

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Preprocessing the training data
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

# Display the shape of the preprocessed data
print(X_train_preprocessed.shape, X_test_preprocessed.shape)

(444575, 2093) (111144, 2093)


In [6]:
# Step 3: Handling Imbalanced Data with SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train_preprocessed, y_train)


In [7]:
# Step 4: Feature Engineering - Apply PCA with fixed components
pca = PCA(n_components=30)
X_train_pca = pca.fit_transform(X_resampled)
X_test_pca = pca.transform(X_test_preprocessed)

In [8]:
# Step 5: Outlier Detection with DBSCAN
dbscan = DBSCAN(eps=0.5, min_samples=5)
outliers = dbscan.fit_predict(X_train_pca)
mask = outliers != -1
X_train_pca = X_train_pca[mask]
y_resampled = y_resampled[mask]

In [9]:
# Step 6: Model Selection and Training with XGBoost
xgb_model = xgb.XGBClassifier(scale_pos_weight=len(y_resampled)/sum(y_resampled), random_state=42)
xgb_model.fit(X_train_pca, y_resampled)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, random_state=42, ...)

In [10]:
# Step 7: Model Evaluation
y_pred = xgb_model.predict(X_test_pca)
y_pred_prob = xgb_model.predict_proba(X_test_pca)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_prob)
precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_pred_prob)
pr_auc = auc(recall_vals, precision_vals)

print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')
print(f'ROC-AUC: {roc_auc:.4f}')
print(f'PR AUC: {pr_auc:.4f}')
print("Classification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.8392
Precision: 0.0224
Recall: 0.9534
F1 Score: 0.0438
ROC-AUC: 0.9677
PR AUC: 0.1679
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.84      0.91    110715
           1       0.02      0.95      0.04       429

    accuracy                           0.84    111144
   macro avg       0.51      0.90      0.48    111144
weighted avg       1.00      0.84      0.91    111144



In [11]:
# Step 8: Model Tuning and Optimization
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [100, 200, 300],
    'scale_pos_weight': [1, len(y_resampled)/sum(y_resampled)]
}

# Grid Search
grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid, scoring='f1', cv=3, verbose=1)
grid_search.fit(X_train_pca, y_resampled)
print("Best parameters found by Grid Search:", grid_search.best_params_)

# Random Search
random_search = RandomizedSearchCV(estimator=xgb_model, param_distributions=param_grid, n_iter=10, scoring='f1', cv=3, verbose=1, random_state=42)
random_search.fit(X_train_pca, y_resampled)
print("Best parameters found by Random Search:", random_search.best_params_)

Fitting 3 folds for each of 54 candidates, totalling 162 fits
Best parameters found by Grid Search: {'learning_rate': 0.2, 'max_depth': 7, 'n_estimators': 300, 'scale_pos_weight': 1}
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best parameters found by Random Search: {'scale_pos_weight': 1, 'n_estimators': 300, 'max_depth': 7, 'learning_rate': 0.2}


In [14]:
# Implementing the Best Hyperparameters
# Train XGBoost model with the best hyperparameters
best_params = {'learning_rate': 0.2, 'max_depth': 7, 'n_estimators': 300, 'scale_pos_weight': 1}
xgb_model = xgb.XGBClassifier(**best_params, random_state=42)
xgb_model.fit(X_train_pca, y_resampled)

# Model Evaluation
y_pred = xgb_model.predict(X_test_pca)
y_pred_prob = xgb_model.predict_proba(X_test_pca)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_prob)
precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_pred_prob)
pr_auc = auc(recall_vals, precision_vals)

print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')
print(f'ROC-AUC: {roc_auc:.4f}')
print(f'PR AUC: {pr_auc:.4f}')
print("Classification Report:")
print(classification_report(y_test, y_pred))


Accuracy: 0.8521
Precision: 0.0237
Recall: 0.9301
F1 Score: 0.0463
ROC-AUC: 0.9638
PR AUC: 0.2231
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.85      0.92    110715
           1       0.02      0.93      0.05       429

    accuracy                           0.85    111144
   macro avg       0.51      0.89      0.48    111144
weighted avg       1.00      0.85      0.92    111144



## trying different methods

In [15]:
# Using ADASYN for Handling Imbalanced Data
from imblearn.over_sampling import ADASYN

# Handling imbalanced data using ADASYN
adasyn = ADASYN(random_state=42)
X_resampled, y_resampled = adasyn.fit_resample(X_train_preprocessed, y_train)

# Applying PCA to reduce dimensions
pca = PCA(n_components=30)
X_train_pca = pca.fit_transform(X_resampled)
X_test_pca = pca.transform(X_test_preprocessed)


In [18]:
# Trying LightGBM Classifier
import lightgbm as lgb

# Train LightGBM model
lgb_model = lgb.LGBMClassifier(scale_pos_weight=len(y_resampled)/sum(y_resampled), random_state=42)
lgb_model.fit(X_train_pca, y_resampled)

# Model Evaluation
y_pred = lgb_model.predict(X_test_pca)
y_pred_prob = lgb_model.predict_proba(X_test_pca)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_prob)
precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_pred_prob)
pr_auc = auc(recall_vals, precision_vals)

print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')
print(f'ROC-AUC: {roc_auc:.4f}')
print(f'PR AUC: {pr_auc:.4f}')
print("Classification Report:")
print(classification_report(y_test, y_pred))


[LightGBM] [Info] Number of positive: 442678, number of negative: 442859
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.024791 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 885537, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499898 -> initscore=-0.000409
[LightGBM] [Info] Start training from score -0.000409
Accuracy: 0.9751
Precision: 0.1303
Recall: 0.9604
F1 Score: 0.2294
ROC-AUC: 0.9950
PR AUC: 0.7812
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.98      0.99    110715
           1       0.13      0.96      0.23       429

    accuracy                           0.98    111144
   macro avg       0.57      0.97      0.61    111144
weighted avg       1.00      0.98      0.98    111144



In [21]:
# Hyperparameter Tuning with Bayesian Optimization
from skopt import BayesSearchCV

# Define hyperparameter search space
param_space = {
    'max_depth': (3, 10),
    'learning_rate': (0.01, 0.3, 'log-uniform'),
    'n_estimators': (100, 1000),
    'scale_pos_weight': (1, 100, 'log-uniform')
}

# Bayesian Optimization
bayes_search = BayesSearchCV(estimator=lgb_model, search_spaces=param_space, scoring='f1', cv=3, n_iter=30, random_state=42, verbose=1)
bayes_search.fit(X_train_pca, y_resampled)
print("Best parameters found by Bayesian Optimization:", bayes_search.best_params_)


Fitting 3 folds for each of 1 candidates, totalling 3 fits
[LightGBM] [Info] Number of positive: 295119, number of negative: 295239
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.019335 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 590358, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499898 -> initscore=-0.000407
[LightGBM] [Info] Start training from score -0.000407
[LightGBM] [Info] Number of positive: 295119, number of negative: 295239
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.018816 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 590358, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499898 -> initscore=-0.000407
[

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped t

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

In [22]:
#training lightGBM with the best parameters
# Best parameters from Bayesian Optimization
best_params = {
    'learning_rate': 0.040343472274915984,
    'max_depth': 8,
    'n_estimators': 940,
    'scale_pos_weight': 4
}

# Train LightGBM model with the best parameters
lgb_model = lgb.LGBMClassifier(**best_params, random_state=42)
lgb_model.fit(X_train_pca, y_resampled)

# Model Evaluation
y_pred = lgb_model.predict(X_test_pca)
y_pred_prob = lgb_model.predict_proba(X_test_pca)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_prob)
precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_pred_prob)
pr_auc = auc(recall_vals, precision_vals)

print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')
print(f'ROC-AUC: {roc_auc:.4f}')
print(f'PR AUC: {pr_auc:.4f}')
print("Classification Report:")
print(classification_report(y_test, y_pred))


[LightGBM] [Info] Number of positive: 442678, number of negative: 442859
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.024997 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 885537, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499898 -> initscore=-0.000409
[LightGBM] [Info] Start training from score -0.000409
Accuracy: 0.9925
Precision: 0.3308
Recall: 0.9114
F1 Score: 0.4854
ROC-AUC: 0.9949
PR AUC: 0.8369
Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      1.00    110715
           1       0.33      0.91      0.49       429

    accuracy                           0.99    111144
   macro avg       0.67      0.95      0.74    111144
weighted avg       1.00      0.99      0.99    111144



The results indicate a significant improvement:

- Accuracy (0.9925): The model correctly predicted 99.25% of the instances.
- Precision (0.3308): Out of all the instances predicted as fraud, 33.08% were actually fraud. This is a notable improvement.
- Recall (0.9114): The model correctly identified 91.14% of all actual fraud cases. This indicates a good balance between recall and precision.
- F1 Score (0.4854): The harmonic mean of precision and recall is 0.4854, indicating a balanced performance.
- ROC-AUC (0.9949): The ROC-AUC score is very high, indicating a strong ability to distinguish between the fraud and non-fraud classes.
- PR AUC (0.8369): The Precision-Recall AUC is also high, reflecting the improved balance between precision and recall.

In [26]:
# implementation of Some Advanced Techniques the model
# Advanced Feature Engineering
# Example of adding interaction terms and polynomial features
from sklearn.preprocessing import PolynomialFeatures

# Add interaction terms and polynomial features
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_resampled_poly = poly.fit_transform(X_resampled)
X_test_poly = poly.transform(X_test_preprocessed)

# Apply PCA to reduce dimensions
pca = PCA(n_components=30)
X_train_pca = pca.fit_transform(X_resampled_poly)
X_test_pca = pca.transform(X_test_poly)


In [27]:
# Using Stacking Classifier
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

# Define base models
base_models = [
    ('xgb', xgb.XGBClassifier(scale_pos_weight=len(y_resampled)/sum(y_resampled), random_state=42)),
    ('lgb', lgb.LGBMClassifier(**best_params, random_state=42))
]

# Define stacking classifier
stack_model = StackingClassifier(estimators=base_models, final_estimator=LogisticRegression())

# Train stacking model
stack_model.fit(X_train_pca, y_resampled)

# Model Evaluation
y_pred = stack_model.predict(X_test_pca)
y_pred_prob = stack_model.predict_proba(X_test_pca)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_prob)
precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_pred_prob)
pr_auc = auc(recall_vals, precision_vals)

print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')
print(f'ROC-AUC: {roc_auc:.4f}')
print(f'PR AUC: {pr_auc:.4f}')
print("Classification Report:")
print(classification_report(y_test, y_pred))


[LightGBM] [Info] Number of positive: 442678, number of negative: 442859
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.024972 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 885537, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499898 -> initscore=-0.000409
[LightGBM] [Info] Start training from score -0.000409
[LightGBM] [Info] Number of positive: 354142, number of negative: 354287
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.019340 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 708429, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499898 -> initscore=-0.000409
[LightGBM] [Info] Start training from score -0.000409
[Light

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

In [24]:
# Using Stacking Classifier
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

# Define base models
base_models = [
    ('xgb', xgb.XGBClassifier(scale_pos_weight=len(y_resampled)/sum(y_resampled), random_state=42)),
    ('lgb', lgb.LGBMClassifier(scale_pos_weight=len(y_resampled)/sum(y_resampled), random_state=42))
]

# Define stacking classifier
stack_model = StackingClassifier(estimators=base_models, final_estimator=LogisticRegression())

# Train stacking model
stack_model.fit(X_train_pca, y_resampled)

# Model Evaluation
y_pred = stack_model.predict(X_test_pca)
y_pred_prob = stack_model.predict_proba(X_test_pca)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_prob)
precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_pred_prob)
pr_auc = auc(recall_vals, precision_vals)

print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1 Score: {f1:.4f}')
print(f'ROC-AUC: {roc_auc:.4f}')
print(f'PR AUC: {pr_auc:.4f}')
print("Classification Report:")
print(classification_report(y_test, y_pred))


[LightGBM] [Info] Number of positive: 442678, number of negative: 442859
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.023577 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 885537, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499898 -> initscore=-0.000409
[LightGBM] [Info] Start training from score -0.000409
[LightGBM] [Info] Number of positive: 354142, number of negative: 354287
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.018210 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 708429, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499898 -> initscore=-0.000409
[LightGBM] [Info] Start training from score -0.000409
[Light